In [22]:
import os
os.chdir('D:\8th semester\Machine Learning Lab')

In [23]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [24]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [25]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

In [26]:
model1 = create_lstm()
model1.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 lstm_4 (LSTM)               (None, 24, 8)             960       
                                                                 
 lstm_5 (LSTM)               (None, 20)                2320      
                                                                 
 flatten_2 (Flatten)         (None, 20)                0         
                                                                 
 dense_2 (Dense)             (None, 1)                 21        
                                                                 
Total params: 3301 (12.89 KB)
Trainable params: 3301 (12.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [27]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [28]:
checkpoints = r'D:\8th semester\Machine Learning Lab\chk\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\8th semester\Machine Learning Lab\chk'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [29]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [30]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [31]:
import os
path_dataset =r'D:\8th semester\Machine Learning Lab\datasets'
path_tr = os.path.join(path_dataset, 'AEP_train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'AEP_test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((84907, 21), (24259, 21), (12130, 21))

In [32]:
time_steps=24
num_features=21

In [33]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.4010317325592041 sec


In [34]:
epochs = 6
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/6
2649/2653 [============================>.] - ETA: 0s - loss: 0.0417 - mae: 0.0417 - mape: 1081.9528
Epoch 1: val_loss improved from inf to 0.02134, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0001-loss0.02.h5
2653/2653 [==============================] - 34s 12ms/step - loss: 0.0417 - mae: 0.0417 - mape: 1080.5122 - val_loss: 0.0213 - val_mae: 0.0213 - val_mape: 10.1428
Epoch 2/6
   6/2653 [..............................] - ETA: 31s - loss: 0.0198 - mae: 0.0198 - mape: 7.4508

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2649/2653 [============================>.] - ETA: 0s - loss: 0.0191 - mae: 0.0191 - mape: 9.9682
Epoch 2: val_loss improved from 0.02134 to 0.01602, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0002-loss0.02.h5
2653/2653 [==============================] - 31s 12ms/step - loss: 0.0191 - mae: 0.0191 - mape: 9.9608 - val_loss: 0.0160 - val_mae: 0.0160 - val_mape: 8.0110
Epoch 3/6
2653/2653 [==============================] - ETA: 0s - loss: 0.0129 - mae: 0.0129 - mape: 187.2053
Epoch 3: val_loss improved from 0.01602 to 0.01084, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0003-loss0.01.h5


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2653/2653 [==============================] - 46s 18ms/step - loss: 0.0129 - mae: 0.0129 - mape: 187.2053 - val_loss: 0.0108 - val_mae: 0.0108 - val_mape: 5.0867
Epoch 4/6
2651/2653 [============================>.] - ETA: 0s - loss: 0.0111 - mae: 0.0111 - mape: 323.4986
Epoch 4: val_loss improved from 0.01084 to 0.00968, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0004-loss0.01.h5
2653/2653 [==============================] - 39s 15ms/step - loss: 0.0111 - mae: 0.0111 - mape: 323.3098 - val_loss: 0.0097 - val_mae: 0.0097 - val_mape: 4.6994
Epoch 5/6
   5/2653 [..............................] - ETA: 33s - loss: 0.0090 - mae: 0.0090 - mape: 3.5730

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2648/2653 [============================>.] - ETA: 0s - loss: 0.0105 - mae: 0.0105 - mape: 92.2819
Epoch 5: val_loss improved from 0.00968 to 0.00912, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0005-loss0.01.h5
2653/2653 [==============================] - 29s 11ms/step - loss: 0.0105 - mae: 0.0105 - mape: 92.1281 - val_loss: 0.0091 - val_mae: 0.0091 - val_mape: 4.0747
Epoch 6/6
   6/2653 [..............................] - ETA: 31s - loss: 0.0100 - mae: 0.0100 - mape: 3.1362

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2652/2653 [============================>.] - ETA: 0s - loss: 0.0097 - mae: 0.0097 - mape: 227.6601
Epoch 6: val_loss improved from 0.00912 to 0.00845, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0006-loss0.01.h5
2653/2653 [==============================] - 28s 11ms/step - loss: 0.0097 - mae: 0.0097 - mape: 227.6125 - val_loss: 0.0085 - val_mae: 0.0085 - val_mape: 3.6732


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [37]:

model = load_model(r'D:\8th semester\Machine Learning Lab\chk\E1-cp-0006-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 2s 3ms/step
Mean Absolute Error (MAE): 135.67
Median Absolute Error (MedAE): 107.98
Mean Squared Error (MSE): 31788.0
Root Mean Squared Error (RMSE): 178.29
Mean Absolute Percentage Error (MAPE): 0.93 %
Median Absolute Percentage Error (MDAPE): 0.75 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [43]:
checkpoints = r'D:\8th semester\Machine Learning Lab\chk\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\8th semester\Machine Learning Lab\chk\E1-cp-0006-loss0.01.h5'
start_epoch= 34

In [44]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th semester\Machine Learning Lab\chk\E1-cp-0006-loss0.01.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [45]:
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/2
2651/2653 [============================>.] - ETA: 0s - loss: 0.0078 - mae: 0.0078 - mape: 227.9616
Epoch 1: val_loss improved from inf to 0.00744, saving model to D:\8th semester\Machine Learning Lab\chk\E1-cp-0001-loss0.01.h5
2653/2653 [==============================] - 33s 12ms/step - loss: 0.0078 - mae: 0.0078 - mape: 227.8286 - val_loss: 0.0074 - val_mae: 0.0074 - val_mape: 3.2380
Epoch 2/2
   1/2653 [..............................] - ETA: 41s - loss: 0.0081 - mae: 0.0081 - mape: 2.4533

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2649/2653 [============================>.] - ETA: 0s - loss: 0.0077 - mae: 0.0077 - mape: 248.5344
Epoch 2: val_loss did not improve from 0.00744
2653/2653 [==============================] - 30s 11ms/step - loss: 0.0077 - mae: 0.0077 - mape: 248.2035 - val_loss: 0.0075 - val_mae: 0.0075 - val_mape: 3.2402


In [46]:

model = load_model(r'D:\8th semester\Machine Learning Lab\chk\E1-cp-0001-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 2s 3ms/step
Mean Absolute Error (MAE): 123.9
Median Absolute Error (MedAE): 94.88
Mean Squared Error (MSE): 27457.99
Root Mean Squared Error (RMSE): 165.7
Mean Absolute Percentage Error (MAPE): 0.85 %
Median Absolute Percentage Error (MDAPE): 0.66 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)
